In [1]:
import numpy as np
import pandas as pd
import pyomo.environ as pyo
import importlib.resources
import sys
import os
import json
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from utils import read_gmlc_gen
import matplotlib.pyplot as plt

## Read gen.csv

In [7]:
csv_path = os.path.join(os.getcwd(), '..', 'Data', 'gen.csv')
json_path = os.path.join(os.getcwd(), '..', 'Data', 'rtsgmle_gen_linear_cost_curve.json')
df = read_gmlc_gen(csv_path)
with open(json_path, 'rb') as f:
    gen_param_dict = json.load(f)

bus_path = os.path.join(os.getcwd(), "..", "Data", "bus.csv")
df_bus = pd.read_csv(bus_path)

In [8]:
# extract the CT, CC, STEAM, WIND, PV units
target_fossil_gen = []
target_renew_gen = []
target_special_gen = []
for idx, row in df.iterrows():
    if row['Unit Type'] in ['CT', 'CC', 'STEAM']:
        target_fossil_gen.append(row)
    if row['Unit Type'] in ['PV', 'WIND', 'HYDRO', 'RTPV']:
        target_renew_gen.append(row)
    if row['Unit Type'] in ['CSP', 'NUCLEAR']:
        target_special_gen.append(row)
df_fossil = pd.DataFrame(target_fossil_gen)
df_renew = pd.DataFrame(target_renew_gen)
df_special = pd.DataFrame(target_special_gen)

In [25]:
df_fossil

,GEN UID,Bus ID,Gen ID,Unit Group,Unit Type,Category,Fuel,MW Inj,MVAR Inj,V Setpoint p.u.,...,Emissions N2O Lbs/MMBTU,Emissions CO Lbs/MMBTU,Emissions VOCs Lbs/MMBTU,Damping Ratio,Inertia MJ/MW,Base MVA,Transformer X p.u.,Unit X p.u.,Pump Load MW,Storage Roundtrip Efficiency
0,101_CT_1,101,1,U20,CT,Oil CT,Oil,8.0,4.96,1.0468,...,0.004,0.11,0.040,0,2.8,24.0,0.13,0.32,0,0
1,101_CT_2,101,2,U20,CT,Oil CT,Oil,8.0,4.96,1.0468,...,0.004,0.11,0.040,0,2.8,24.0,0.13,0.32,0,0
2,101_STEAM_3,101,3,U76,STEAM,Coal,Coal,76.0,0.14,1.0468,...,0.004,0.02,0.003,0,3.0,89.0,0.13,0.30,0,0
3,101_STEAM_4,101,4,U76,STEAM,Coal,Coal,76.0,0.14,1.0468,...,0.004,0.02,0.003,0,3.0,89.0,0.13,0.30,0,0
4,102_CT_1,102,1,U20,CT,Oil CT,Oil,8.0,4.88,1.0467,...,0.004,0.11,0.040,0,2.8,24.0,0.13,0.32,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67,321_CC_1,321,1,U355,CC,Gas CC,NG,355.0,-3.34,1.0500,...,0.000,0.00,0.000,0,5.0,414.0,0.13,0.30,0,0
68,322_CT_5,322,5,U55,CT,Gas CT,NG,55.0,-9.73,1.0500,...,0.000,0.00,0.000,0,2.8,64.0,0.13,0.32,0,0
69,322_CT_6,322,6,U55,CT,Gas CT,NG,55.0,-9.73,1.0500,...,0.000,0.00,0.000,0,2.8,64.0,0.13,0.32,0,0
70,323_CC_1,323,1,U355,CC,Gas CC,NG,355.0,37.41,1.0500,...,0.000,0.00,0.000,0,5.0,414.0,0.13,0.30,0,0


In [26]:
df_renew

,GEN UID,Bus ID,Gen ID,Unit Group,Unit Type,Category,Fuel,MW Inj,MVAR Inj,V Setpoint p.u.,...,Emissions N2O Lbs/MMBTU,Emissions CO Lbs/MMBTU,Emissions VOCs Lbs/MMBTU,Damping Ratio,Inertia MJ/MW,Base MVA,Transformer X p.u.,Unit X p.u.,Pump Load MW,Storage Roundtrip Efficiency
74,122_HYDRO_1,122,1,U50,HYDRO,Hydro,Hydro,50.0,-6.79,1.05,...,0.0,0.0,0.0,0,3.5,53.0,0.1,0.28,0,0
75,122_HYDRO_2,122,2,U50,HYDRO,Hydro,Hydro,50.0,-6.79,1.05,...,0.0,0.0,0.0,0,3.5,53.0,0.1,0.28,0,0
76,122_HYDRO_3,122,3,U50,HYDRO,Hydro,Hydro,50.0,-6.79,1.05,...,0.0,0.0,0.0,0,3.5,53.0,0.1,0.28,0,0
77,122_HYDRO_4,122,4,U50,HYDRO,Hydro,Hydro,50.0,-6.79,1.05,...,0.0,0.0,0.0,0,3.5,53.0,0.1,0.28,0,0
78,122_HYDRO_5,122,5,U50,HYDRO,Hydro,Hydro,50.0,-6.79,1.05,...,0.0,0.0,0.0,0,3.5,53.0,0.1,0.28,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
152,213_RTPV_1,213,1,RTPV,RTPV,Solar RTPV,Solar,0.0,0.00,1.00,...,0.0,0.0,0.0,0,0.0,13.2,0.0,0.00,0,0
153,309_WIND_1,309,1,WIND,WIND,Wind,Wind,0.0,0.00,1.00,...,0.0,0.0,0.0,0,0.0,148.3,0.0,0.00,0,0
154,317_WIND_1,317,1,WIND,WIND,Wind,Wind,0.0,0.00,1.00,...,0.0,0.0,0.0,0,0.0,799.1,0.0,0.00,0,0
155,303_WIND_1,303,1,WIND,WIND,Wind,Wind,0.0,0.00,1.00,...,0.0,0.0,0.0,0,0.0,847.0,0.0,0.00,0,0


In [27]:
df_special

,GEN UID,Bus ID,Gen ID,Unit Group,Unit Type,Category,Fuel,MW Inj,MVAR Inj,V Setpoint p.u.,...,Emissions N2O Lbs/MMBTU,Emissions CO Lbs/MMBTU,Emissions VOCs Lbs/MMBTU,Damping Ratio,Inertia MJ/MW,Base MVA,Transformer X p.u.,Unit X p.u.,Pump Load MW,Storage Roundtrip Efficiency
73,121_NUCLEAR_1,121,1,U400,NUCLEAR,Nuclear,Nuclear,400.0,-21.87,1.05,...,0.0,0.0,0.0,0,5.0,471.0,0.15,0.4,0,0
116,212_CSP_1,212,1,CSP,CSP,CSP,Solar,0.0,0.00,1.00,...,0.0,0.0,0.0,0,0.0,200.0,0.00,0.0,0,0


## calculate cost for the start up with different suitations

In [28]:
for idx, row in df_fossil.iterrows():
    fuel_price = row['Fuel Price $/MMBTU']
    cold_start_energy = row['Start Heat Cold MBTU']
    warm_start_energy = row['Start Heat Warm MBTU']
    hot_start_energy = row['Start Heat Hot MBTU']
    cold_start_cost = fuel_price * cold_start_energy
    warm_start_cost = fuel_price * warm_start_energy
    hot_start_cost =  fuel_price * hot_start_energy

In [29]:
# generate a dictionary contains fossil generator information
fossil_dict = {}
for idx, row in df_fossil.iterrows():
    gen_name = row['GEN UID']
    gen_dict = {}
    gen_dict['name'] = gen_name
    bus_id= row['Bus ID']
    gen_dict['bus_name'] = df_bus[df_bus["Bus ID"]==bus_id]["Bus Name"].to_list()[0]
    gen_dict['gen_type'] = row['Unit Type']
    gen_dict['max_p'] = row['PMax MW']
    gen_dict['min_p'] = row['PMin MW']
    gen_dict['ramp'] = row['Ramp Rate MW/Min'] * 60 # ramp rate should be MW/hr
    gen_dict['fuel_p'] = row['Fuel Price $/MMBTU']
    gen_dict['min_down_time'] = int(np.round(row['Min Down Time Hr'], 0)) # should be rounded to an integer
    gen_dict['min_up_time'] = int(np.round(row['Min Up Time Hr'], 0)) # should be rounded to an integer
    gen_dict['start_up_time_hot'] = int(np.round(row['Start Time Hot Hr'], 0))
    gen_dict['start_up_time_warm'] = int(np.round(row['Start Time Warm Hr'], 0))
    gen_dict['start_up_time_cold'] = int(np.round(row['Start Time Cold Hr'], 0))
    gen_dict['start_heat_hot'] = row['Start Heat Hot MBTU']
    gen_dict['start_heat_warm'] = row['Start Heat Warm MBTU']
    gen_dict['start_heat_cold'] = row['Start Heat Cold MBTU']
    gen_dict['cost_curve'] = gen_param_dict[gen_name]
    
    fossil_dict[gen_name] = gen_dict

In [30]:
# generate a dictionary contains renewable generator information
renew_dict = {}
for idx, row in df_renew.iterrows():
    gen_name = row['GEN UID']
    gen_dict = {}
    gen_dict['name'] = gen_name
    gen_dict['gen_type'] = row['Unit Type']
    bus_id= row['Bus ID']
    gen_dict['bus_name'] = df_bus[df_bus["Bus ID"]==bus_id]["Bus Name"].to_list()[0]
    gen_dict['max_p'] = row['PMax MW']
#     gen_dict['min_p'] = row['PMin MW']
#     gen_dict['ramp'] = row['Ramp Rate MW/Min']
#     gen_dict['min_down_time'] = row['Min Down Time Hr']
#     gen_dict['min_up_time'] = row['Min Up Time Hr']
#     gen_dict['start_up_time_hot'] = row['Start Time Hot Hr']
#     gen_dict['start_up_time_cold'] = row['Start Time Cold Hr']
#     gen_dict['start_up_time_cold'] = row['Start Time Warm Hr']
    gen_dict['cost_curve'] = {'slope': 0, 'intercept': 0} # for the renewable generators, the operation cost is 0.
    
    renew_dict[gen_name] = gen_dict

In [31]:
 renew_dict["303_WIND_1"]

{'name': '303_WIND_1',
 'gen_type': 'WIND',
 'bus_name': 'Caesar',
 'max_p': 847.0,
 'cost_curve': {'slope': 0, 'intercept': 0}}

## Generator startup type check

In [32]:
# check the generator types to decide its startup types
def gen_startup_cost(type_gen_dict, gen_name):
#     min_down_time = type_gen_dict[gen_name]['min_down_time']
    hot_time = type_gen_dict[gen_name]['start_up_time_hot']
    warm_time = type_gen_dict[gen_name]['start_up_time_warm']
    cold_time = type_gen_dict[gen_name]['start_up_time_cold']
    fuel_p = type_gen_dict[gen_name]['fuel_p']
    start_heat_hot = type_gen_dict[gen_name]['start_heat_hot']
    start_heat_warm = type_gen_dict[gen_name]['start_heat_warm']
    start_heat_cold = type_gen_dict[gen_name]['start_heat_cold']
    
    start_up_cost_hot = start_heat_hot*fuel_p
    start_up_cost_warm = start_heat_warm*fuel_p
    start_up_cost_cold = start_heat_cold*fuel_p
    start_up_cost = {'hot': start_up_cost_hot, 'warm': start_up_cost_warm, 'cold': start_up_cost_cold}
    
    return start_up_cost

In [34]:
all_gen_dict = {}
all_gen_dict['fossil'] = fossil_dict
all_gen_dict['renew'] = renew_dict
# all_gen_dict['special'] = special_dict

gen_dict_path = os.path.join(os.getcwd(), '..', 'Data', 'gen_dict.json')

with open(gen_dict_path, "w") as f:
    json.dump(all_gen_dict, f)
    print("Successfully saved generator parameters to json files")

Successfully saved generator parameters to json files


In [90]:
fossil_dict['101_CT_1']

{'name': '101_CT_1',
 'fuel_p': 10.3494,
 'gen_type': 'CT',
 'max_p': 20.0,
 'min_p': 8,
 'ramp': 180.0,
 'min_down_time': 1.0,
 'min_up_time': 1.0,
 'start_up_time_hot': 0.0,
 'start_up_time_warm': 0.0,
 'start_up_time_cold': 1,
 'start_heat_hot': 5.0,
 'start_heat_warm': 5.0,
 'start_heat_cold': 5.0,
 'cost_curve': {'slope': 100.728, 'intercept': 272.4442528000002},
 'duration_type': 'C'}

## Test RHPTForecaster

In [1]:
from idaes.apps.grid_integration.forecaster import RHPTForecaster
import numpy as np

In [2]:
p = []
for i in range(10):
    p.append([i]*24)
price = np.array(p).reshape(-1)

scenario = 5
horizon = 36
planning_horizon = 24

In [3]:
forecaster = RHPTForecaster(price, scenario, horizon, planning_horizon)

pointer=0
forecaster._forecast_prices(pointer)

2025-07-02 15:48:13 [INFO] idaes.apps.grid_integration.forecaster: Number of periods from provided data is 10.
2025-07-02 15:48:13 [INFO] idaes.apps.grid_integration.forecaster: Number of scenarios is 5.
2025-07-02 15:48:13 [INFO] idaes.apps.grid_integration.forecaster: The length of the scenario is 36.


array([[5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5,
        5, 5, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6],
       [6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6,
        6, 6, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7],
       [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7,
        7, 7, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8],
       [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8,
        8, 8, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9],
       [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9,
        9, 9, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])

In [7]:
csv_path = "../Data/all_bus_lmp.csv"
df_lmp = pd.read_csv(csv_path)
lmp_arr = df_lmp['Abel_LMP'].to_numpy()
len(lmp_arr)

8784

In [8]:
forecaster = RHPTForecaster(lmp_arr, scenario, horizon, planning_horizon)

2025-07-02 15:48:25 [INFO] idaes.apps.grid_integration.forecaster: Number of periods from provided data is 366.
2025-07-02 15:48:25 [INFO] idaes.apps.grid_integration.forecaster: Number of scenarios is 5.
2025-07-02 15:48:25 [INFO] idaes.apps.grid_integration.forecaster: The length of the scenario is 36.


In [10]:
pointer = 7
lmp_forecasted = forecaster._forecast_prices(pointer)
for idx, l in enumerate(lmp_forecasted):
    actual_l = lmp_arr[(2+idx)*24: (2+idx)*24 + horizon]
    print(len(l), l- actual_l)

36 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
36 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
36 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
36 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
36 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


## Test Rolling Horizon PT Results

In [2]:
from utils import read_rolling_horizon_params, check_optimal_status, calculate_total_profit,calculate_total_startup_shutdown
from Rolling_horizon.fossil_rolling_horizon_PT_parameter import gen_dict
from idaes.apps.grid_integration import RHPTForecaster

In [4]:
# read the results dictionary
json_path = os.path.join(os.getcwd(), "..", "Data", "test_366_gen_101_STEAM_3_result.json")
res_dict = read_rolling_horizon_params(json_path)

In [4]:
# check if all period problem converged to optimal
non_optimal = check_optimal_status(res_dict)
print(non_optimal)

[]


In [5]:
actual_profit = calculate_total_profit(res_dict)
ideal_profit = calculate_total_profit(res_dict, name="IdeaProfit")

print(f"PCM Profit: {17.63} M$")
print(f"Standard Profit: {19.11} M$.")
print(f"Ideal Profit: {np.round(ideal_profit/1e6, 2)} M$.")
print(f"Actual Profit: {np.round(actual_profit/1e6, 2)} M$.")

PCM Profit: 17.63 M$
Standard Profit: 19.11 M$.
Ideal Profit: 17.28 M$.
Actual Profit: 14.22 M$.


In [27]:
print("ideal", "actual")
for key in res_dict.keys():
    print(f"{np.round(res_dict[key]['IdeaProfit'], 2)}.", f"{np.round(res_dict[key]['ActualProfit'], 2)}.")

ideal actual
6236.44. -23544.4.
11888.46. 4400.1.
15582.73. -2490.63.
10249.28. 6191.78.
3415.52. -18961.42.
-3386.75. 3793.56.
7097.85. -6481.91.
-11558.45. -29750.33.
-3130.37. -15719.3.
0.0. 0.0.
-13499.95. -9562.88.
-14676.25. -13434.56.
-14709.48. -8615.19.
-12442.61. -19095.22.
-46.71. -10027.18.
117.24. -13811.83.
0.0. 0.0.
-12354.63. -9623.83.
-12537.97. -7397.53.
-11627.24. 3060.76.
4984.85. 177879.16.
44565.51. 1078.66.
46645.69. -20616.42.
43439.64. -16019.7.
34607.79. -19184.17.
31570.49. -11283.92.
-13440.15. -17391.65.
-15721.85. -17266.23.
-15858.04. -16590.77.
-16550.22. -16605.81.
-16982.1. -16640.85.
-16755.19. -17233.18.
155087.95. -16076.27.
169757.9. 125969.66.
234192.77. 24571.16.
241746.07. 170802.19.
282507.6. -2324.59.
104556.72. 9543.45.
106177.43. 13462.77.
45248.05. -4536.39.
39287.44. 765421.86.
156584.03. 180903.41.
193078.15. -2254.07.
190567.16. 4508.29.
188808.62. 1211.95.
189958.29. -1114.62.
36689.61. 5775.17.
2687.98. 632963.94.
129853.06. 219086.74.

In [28]:
for key in res_dict.keys():
    total_shutdown = sum(res_dict[key]["gen_101_STEAM_3"]["OperationVariables_shutdown"]["1"].values())
    total_startup = sum(res_dict[key]["gen_101_STEAM_3"]["OperationVariables_startup"]["1"].values())
    if total_shutdown > 0:
        print("shutdown", key)
    if total_startup > 0:
        print("startup", key)

startup period_0
shutdown period_6
startup period_7
shutdown period_9
shutdown period_10
startup period_10
shutdown period_11
startup period_11
shutdown period_12
startup period_12
startup period_13
shutdown period_16
shutdown period_17
startup period_17
shutdown period_18
startup period_18
startup period_19
shutdown period_25
shutdown period_26
startup period_26
shutdown period_27
startup period_27
shutdown period_28
startup period_28
shutdown period_29
startup period_29
shutdown period_30
startup period_30
shutdown period_31
startup period_31
startup period_32
shutdown period_54
shutdown period_55
startup period_55
startup period_56
shutdown period_71
startup period_72
shutdown period_79
startup period_80
shutdown period_107
shutdown period_108
startup period_108
shutdown period_109
startup period_109
shutdown period_110
startup period_110
shutdown period_111
startup period_111
shutdown period_112
startup period_112
shutdown period_113
startup period_113
startup period_114
shutdown p

In [29]:
calculate_total_startup_shutdown(res_dict, gen_name="gen_101_STEAM_3")

(71.0, 70.0)

In [22]:
slope = 16.466, 
intercept = 331.4380855711627

lmp_path = os.path.join("..", "Data", "all_bus_lmp.csv")
df_lmp = pd.read_csv(lmp_path)
lmp_data = df_lmp[gen_dict["gen_101_STEAM_3"]["bus_name"]+"_LMP"].to_numpy()

forecaster = RHPTForecaster(price_signal=lmp_data,
                            scenario=5,
                            horizon=36,
                            planning_horizon=24)

elec_rev = 0
cost = 0
pointer = 0

for i in res_dict:
    power =  res_dict[i]['gen_101_STEAM_3']['OperationVariables_power']['1']
    op_mode = res_dict[i]['gen_101_STEAM_3']['OperationVariables_op_mode']['1']
    power_arr = np.array([v for v in power.values()])
    op_mode_arr = np.array([v for v in op_mode.values()])
    lmp = forecaster.fetch_original_signal(pointer)
    cost += sum(power_arr * slope + op_mode_arr * intercept)
    elec_rev += sum(lmp*power_arr)
    pointer += 1
    
cost += 71*5284.8*2.11399

elec_rev, cost, elec_rev-cost


2025-08-08 17:22:25 [INFO] idaes.apps.grid_integration.forecaster: Number of periods from provided data is 366.
2025-08-08 17:22:25 [INFO] idaes.apps.grid_integration.forecaster: Number of scenarios is 5.
2025-08-08 17:22:25 [INFO] idaes.apps.grid_integration.forecaster: The length of the scenario is 36.


(25360640.22043799, 11141318.534188543, 14219321.686249446)

## Using new LMP

### PCM

In [32]:
df_lmp = pd.read_csv("Bus_LMP.csv")
df_dispatch = pd.read_csv("Generator_Dispatch.csv")

state_arr = df_dispatch["101_STEAM_3_Unit State"].to_numpy()

int_state_arr = []
for i in state_arr:
    if i:
        int_state_arr.append(1)
    else:
        int_state_arr.append(0)
int_state_arr = np.array(int_state_arr)

startups = 0
for i in range(1, len(state_arr)):
    if int(state_arr[i]) - int(state_arr[i-1]) == 1:
        startups += 1
        
dispatch_arr = df_dispatch["101_STEAM_3_Dispatch"].to_numpy()

costs = dispatch_arr * 16.466 + 331.4381*int_state_arr
sum(costs) + 2.11399*5284.8*6

# sum(df_lmp["Abel_LMP"].to_numpy() * dispatch_arr)

12467569.950188339

### Determinstic PT

In [33]:
js_path = "../Data/det_fossil_PT_fixed_dispatch_results.json"

with open(js_path, "r") as f:
    res_dict = json.load(f)

startup = 0
rev = 0
vom = 0
for i in range(1, 366*24):
    startup += res_dict[str(i)]['startup']
    rev += res_dict[str(i)]['rev']
    vom += res_dict[str(i)]['vom']
    
startup, rev, vom + 2.11399*5284.8*6

(7.0, 29073470.33426585, 12465986.972160637)